In [1]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

## Converting Data Types and formats

In [2]:
df = pl.scan_parquet("/home/ma/a/alb25/Project/thesis_code/data/raw/*")

In [3]:
# Adding the user type columns

def add_user_type(column):
    return (pl.when(pl.col(column).str.contains(r"^U\d+@")).then(pl.lit("human"))
                    .when(pl.col(column).str.contains(r"^C\d+\$@")).then(pl.lit("machine"))
                    .when(pl.col(column).str.contains(r"^(SYSTEM|LOCAL SERVICE|NETWORK SERVICE)@")).then(pl.lit("system"))
                    .when(pl.col(column).str.contains(r"^ANONYMOUS LOGON@")).then(pl.lit("anon"))
                    .otherwise(pl.lit("other")))

df = df.with_columns([add_user_type("source_user@domain").alias("source_user_type"),
    add_user_type("destination_user@domain").alias("destination_user_type")])

In [4]:
# converting the tiem column to a good format being a duration
# EDA revealed 58 days of data
df = df.with_columns(pl.col('time').cast(pl.Int64).alias('time'))
df = df.with_columns(pl.duration(seconds=pl.col('time')).alias('time'))

In [5]:
# Converting the dataframe success column to a boolean
# EDA revealed that it can only take 2 values Success and Failure
df = df.with_columns(pl.when(pl.col('success/failure') == 'Success').then(True).otherwise(False).alias('success')).drop('success/failure')

In [6]:
# Converting columns to categorical
for column_name in ('authentication_orientation', 'authentication_type', 'logon_type', 'source_user_type', 'destination_user_type'):
    col_vals = df.select(column_name).unique().collect().to_series().to_list()
    col_vals = pl.Enum(col_vals)
    df = df.with_columns(pl.col(column_name).cast(col_vals).alias(column_name))

### Creating a train test validaiton split

2 week for test 1 week validation the rest for train

This was used for the EDA

In [7]:
df = df.with_columns(pl.col('time').dt.total_days().alias('day'))

In [10]:
test_df = df.filter(pl.col('day') >= 44)
validation_df = df.filter((pl.col('day') >= 37) & (pl.col('day') < 44))
train_df = df.filter((pl.col('day') < 37) & (pl.col('day') >= 2))

In [11]:
for df_name, d in {'test_df' : test_df, 'train_df' : train_df, 'validation_df' :validation_df}.items():
    d = d.drop('day')
    d = d.sort(['source_user@domain', 'time'])
    d.sink_parquet(f'/home/ma/a/alb25/Project/thesis_code/data/processed/source/{df_name}.parquet',
                   compression="lz4", statistics=True, row_group_size=250_000)


### Filtering to machine and human users

In [12]:
import polars as pl 

df = pl.scan_parquet('/home/ma/a/alb25/Project/thesis_code/data/processed/source/train_df.parquet')

In [ ]:
cut_off = 200
df = df.filter(pl.col('source_user_type').is_in(['human', 'machine']))
wanted_users = df.group_by('source_user@domain').agg(pl.col('source_user@domain').len().alias('count')).filter(
            pl.col('count') > cut_off).select('source_user@domain')

In [14]:
# Adding in the row_id column and storing data
def create_datasets(df, df_name, wanted_users=wanted_users):
    df = df.join(wanted_users, how='semi', on='source_user@domain')
    df = df.sort(by=['source_user@domain', 'time', 'destination_user@domain']).with_row_index(name='row_id')
    df.sink_parquet(f'/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate/input/{df_name}_w_metadata.parquet', compression='zstd', compression_level=8)
    df = df.select(['source_user@domain', 'time', 'row_id'])
    df.sink_parquet(f'/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate/input/{df_name}.parquet', compression="lz4", statistics=True, row_group_size=250_000)

In [15]:
## Writing the train dataset
create_datasets(df, 'train_df')

In [16]:
# Reading and writing the validation dataset
df = pl.scan_parquet('/home/ma/a/alb25/Project/thesis_code/data/processed/source/validation_df.parquet')
create_datasets(df, 'validation_df')

In [ ]:
# Reading and writing the test 
df = pl.scan_parquet('/home/ma/a/alb25/Project/thesis_code/data/processed/source/test_df.parquet')
create_datasets(df, 'test_df')